# Deepfake Toolkit Standalone Walkthrough

Run only the section you need. Each pipeline (classification or segmentation) is self-contained and redefines
all helpers, so there is no hidden dependency between sections.


---
## Part A — Classification Pipeline


### A.1 Prerequisites & Imports
Install: `torch`, `torchvision`, `datasets`, `scikit-learn`, `matplotlib`, `seaborn`.
This cell also seeds RNGs so results are reproducible.


In [ ]:
# Classification imports and helpers
import json
import math
import random
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch import amp
from torchvision.transforms import v2 as T
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix


def cls_set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cls_timestamp() -> str:
    return datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')


def cls_device() -> torch.device:
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')



### A.2 Configuration & Run Directory
Edit the knobs below to tailor dataset size, augmentation, or epochs. Each run writes artefacts to a timestamped folder.


In [ ]:
CLASS_CFG = {
    'dataset_name': 'saberzl/SID_Set',
    'image_size': 128,
    'train_samples': 160,
    'val_samples': 64,
    'test_samples': 64,
    'batch_size': 16,
    'num_workers': 0,
    'learning_rate': 1e-3,
    'epochs': 15,
    'label_smoothing': 0.05,
    'grad_clip_norm': 2.0,
    'ema_decay': 0.995,
    'use_cosine_schedule': True,
    'augment': True,
    'random_erasing_prob': 0.25,
    'seed': 42,
    'output_root': Path('standalone_outputs_classification'),
}
CLASS_CFG['output_root'].mkdir(parents=True, exist_ok=True)
cls_set_seed(CLASS_CFG['seed'])
CLASS_DEVICE = cls_device()
CLASS_RUN_ID = cls_timestamp()
CLASS_RUN_DIR = CLASS_CFG['output_root'] / CLASS_RUN_ID
CLASS_RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Classification configuration:')
print(json.dumps({k: (str(v) if isinstance(v, Path) else v) for k, v in CLASS_CFG.items()}, indent=2))
print('Device:', CLASS_DEVICE)
print('Output folder:', CLASS_RUN_DIR)



### A.3 Data Preparation
We define augmentation transforms, wrap SID_Set records, and build DataLoaders.


In [ ]:
CLASS_NAMES = ['Real', 'Synthetic', 'Tampered']


def class_transforms(image_size: int, augment: bool, random_erasing_prob: float):
    ops = []
    if augment:
        ops.extend([
            T.RandomResizedCrop(image_size, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomApply([T.RandomPerspective(distortion_scale=0.08)], p=0.3),
            T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.04),
            T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.3),
        ])
    else:
        ops.append(T.Resize((image_size, image_size), antialias=True))
    ops.extend([
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    if augment and random_erasing_prob > 0:
        ops.append(T.RandomErasing(p=random_erasing_prob, scale=(0.02, 0.2), ratio=(0.3, 3.3)))
    return T.Compose(ops)


class ClassificationDataset(Dataset):
    def __init__(self, hf_dataset, image_size: int, augment: bool, random_erasing_prob: float):
        self.dataset = hf_dataset
        self.transform = class_transforms(image_size, augment, random_erasing_prob)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        record = self.dataset[idx]
        image = record['image'].convert('RGB')
        tensor = self.transform(image)
        label = torch.tensor(int(record['label']), dtype=torch.long)
        return {'image': tensor, 'label': label}


def build_class_loaders(cfg):
    hf_train = load_dataset(cfg['dataset_name'], split=f"train[:{cfg['train_samples']}]", streaming=False)
    hf_val = load_dataset(cfg['dataset_name'], split=f"validation[:{cfg['val_samples']}]", streaming=False)
    hf_test = load_dataset(cfg['dataset_name'], split=f"validation[{cfg['val_samples']}:{cfg['val_samples']+cfg['test_samples']}]", streaming=False)

    train_ds = ClassificationDataset(hf_train, cfg['image_size'], augment=cfg['augment'], random_erasing_prob=cfg['random_erasing_prob'])
    val_ds = ClassificationDataset(hf_val, cfg['image_size'], augment=False, random_erasing_prob=0.0)
    test_ds = ClassificationDataset(hf_test, cfg['image_size'], augment=False, random_erasing_prob=0.0)

    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, num_workers=cfg['num_workers'])
    val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'])
    test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'])
    return train_loader, val_loader, test_loader

train_loader_c, val_loader_c, test_loader_c = build_class_loaders(CLASS_CFG)
print('Classification dataloaders -> train:', len(train_loader_c), 'val:', len(val_loader_c), 'test:', len(test_loader_c))



### A.4 Visualise Augmented Samples

In [ ]:
batch = next(iter(train_loader_c))
grid = vutils.make_grid(batch['image'][:8], nrow=4, normalize=True)
plt.figure(figsize=(6,6))
plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
plt.axis('off')
plt.title('Classification – Augmented Samples')
plt.show()



### A.5 Model & Optimiser

In [ ]:
class Classifier(nn.Module):
    def __init__(self, base_width=32, num_classes=3):
        super().__init__()
        widths = [base_width, base_width*2, base_width*4]
        self.stem = self._block(3, widths[0])
        self.stage1 = self._block(widths[0], widths[1])
        self.stage2 = self._block(widths[1], widths[2])
        self.pool = nn.MaxPool2d(2)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(widths[2], widths[2]),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(widths[2], num_classes)
        )

    @staticmethod
    def _block(in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.pool(self.stem(x))
        x = self.pool(self.stage1(x))
        x = self.stage2(x)
        return self.head(x)

model_c = Classifier(num_classes=len(CLASS_NAMES)).to(CLASS_DEVICE)
criterion_c = nn.CrossEntropyLoss(label_smoothing=CLASS_CFG['label_smoothing'])
optimizer_c = optim.AdamW(model_c.parameters(), lr=CLASS_CFG['learning_rate'], weight_decay=1e-4)
scaler_c = amp.amp.GradScaler('cuda', 'cuda', enabled=(CLASS_DEVICE.type == 'cuda'))
scheduler_c = optim.lr_scheduler.CosineAnnealingLR(optimizer_c, T_max=CLASS_CFG['epochs']) if CLASS_CFG['use_cosine_schedule'] else None
ema_c = Classifier(num_classes=len(CLASS_NAMES)).to(CLASS_DEVICE)
ema_c.load_state_dict(model_c.state_dict())
for param in ema_c.parameters():
    param.requires_grad_(False)



### A.6 Training Loop

In [ ]:
history_c = defaultdict(list)
best_state_c = None
best_val_acc = -math.inf

for epoch in range(CLASS_CFG['epochs']):
    model_c.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch in train_loader_c:
        images = batch['image'].to(CLASS_DEVICE)
        labels = batch['label'].to(CLASS_DEVICE)

        optimizer_c.zero_grad(set_to_none=True)
        with amp.autocast(device_type='cuda', enabled=(CLASS_DEVICE.type == 'cuda')):
            logits = model_c(images)
            loss = criterion_c(logits, labels)

        scaler_c.scale(loss).backward()
        if CLASS_CFG['grad_clip_norm'] > 0:
            scaler_c.unscale_(optimizer_c)
            torch.nn.utils.clip_grad_norm_(model_c.parameters(), CLASS_CFG['grad_clip_norm'])
        scaler_c.step(optimizer_c)
        scaler_c.update()

        if CLASS_CFG['ema_decay'] > 0:
            for ema_param, param in zip(ema_c.parameters(), model_c.parameters()):
                ema_param.data.mul_(CLASS_CFG['ema_decay']).add_(param.data, alpha=1 - CLASS_CFG['ema_decay'])

        train_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += images.size(0)

    if scheduler_c is not None:
        scheduler_c.step()

    model_c.eval()
    ema_c.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in val_loader_c:
            images = batch['image'].to(CLASS_DEVICE)
            labels = batch['label'].to(CLASS_DEVICE)
            logits = ema_c(images)
            loss = criterion_c(logits, labels)
            val_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total
    val_loss /= max(val_total, 1)
    val_acc = val_correct / max(val_total, 1)

    history_c['train_loss'].append(train_loss)
    history_c['train_acc'].append(train_acc)
    history_c['val_loss'].append(val_loss)
    history_c['val_acc'].append(val_acc)
    history_c['lr'].append(optimizer_c.param_groups[0]['lr'])

    print(f"[Classification] Epoch {epoch+1:02d}/{CLASS_CFG['epochs']} - train_loss={train_loss:.3f} train_acc={train_acc:.3f} val_loss={val_loss:.3f} val_acc={val_acc:.3f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_c = {k: v.clone() for k, v in ema_c.state_dict().items()}

if best_state_c is not None:
    ema_c.load_state_dict(best_state_c)



### A.7 Evaluation & Visualisations

In [ ]:
model_path_c = CLASS_RUN_DIR / 'best_model.pth'
history_path_c = CLASS_RUN_DIR / 'history.json'
torch.save(ema_c.state_dict(), model_path_c)
with history_path_c.open('w') as fp:
    json.dump({k: v for k, v in history_c.items()}, fp, indent=2)
print('Saved classification model to', model_path_c)

ema_c.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader_c:
        images = batch['image'].to(CLASS_DEVICE)
        labels = batch['label'].to(CLASS_DEVICE)
        logits = ema_c(images)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
cm = confusion_matrix(all_labels, all_preds)
(CLASS_RUN_DIR / 'evaluation.json').write_text(json.dumps({'classification_report': report, 'confusion_matrix': cm.tolist()}, indent=2))
print('Classification macro F1:', report['macro avg']['f1-score'])

plt.figure(figsize=(8,4))
plt.plot(history_c['train_loss'], label='Train')
plt.plot(history_c['val_loss'], label='Validation')
plt.title('Classification Loss Curves')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()

plt.figure(figsize=(8,4))
plt.plot(history_c['train_acc'], label='Train')
plt.plot(history_c['val_acc'], label='Validation')
plt.title('Classification Accuracy Curves')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.show()

plt.figure(figsize=(8,4))
plt.plot(history_c['lr'])
plt.title('Classification Learning Rate Schedule')
plt.xlabel('Epoch'); plt.ylabel('LR'); plt.show()

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Classification Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.show()

batch = next(iter(test_loader_c))
images = batch['image'][:6].to(CLASS_DEVICE)
labels = batch['label'][:6]
with torch.no_grad():
    logits = ema_c(images)
    preds = logits.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 3, figsize=(9,6))
for ax, img, pred, label in zip(axes.flatten(), images.cpu(), preds, labels):
    img_disp = img * torch.tensor([0.229, 0.224, 0.225]).view(3,1,1) + torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    ax.imshow(np.transpose(img_disp.clamp(0,1).numpy(), (1,2,0)))
    ax.set_title(f"Pred: {CLASS_NAMES[pred]}
True: {CLASS_NAMES[label]}")
    ax.axis('off')
plt.tight_layout(); plt.show()

print('Classification artifacts:')
print('
'.join(sorted(p.name for p in CLASS_RUN_DIR.iterdir())))



---
## Part B — Segmentation Pipeline


### B.1 Prerequisites & Imports
This block re-imports everything it needs so it can run independently of the classification cells.


In [ ]:
import json
import math
import random
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch import amp
from torchvision.transforms import functional as F
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset


def seg_set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seg_timestamp() -> str:
    return datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')


def seg_device() -> torch.device:
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')



### B.2 Configuration & Run Directory

In [ ]:
SEG_CFG = {
    'dataset_name': 'saberzl/SID_Set',
    'image_size': 256,
    'train_samples': 220,
    'val_samples': 80,
    'test_samples': 80,
    'batch_size': 8,
    'num_workers': 0,
    'learning_rate': 6e-4,
    'epochs': 18,
    'grad_clip_norm': 1.5,
    'ema_decay': 0.99,
    'use_cosine_schedule': True,
    'augment': True,
    'seed': 1337,
    'output_root': Path('standalone_outputs_segmentation'),
}
SEG_CFG['output_root'].mkdir(parents=True, exist_ok=True)
seg_set_seed(SEG_CFG['seed'])
SEG_DEVICE = seg_device()
SEG_RUN_ID = seg_timestamp()
SEG_RUN_DIR = SEG_CFG['output_root'] / SEG_RUN_ID
SEG_RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Segmentation configuration:')
print(json.dumps({k: (str(v) if isinstance(v, Path) else v) for k, v in SEG_CFG.items()}, indent=2))
print('Device:', SEG_DEVICE)
print('Output folder:', SEG_RUN_DIR)



### B.3 Dataset & Augmentation

In [ ]:
@dataclass
class SegAugCfg:
    image_size: int
    enable: bool


def seg_random_crop_pair(image, mask, size, scale=(0.6, 1.0), ratio=(0.75, 1.33)):
    for _ in range(10):
        area = image.width * image.height
        target_area = area * random.uniform(*scale)
        aspect_ratio = math.exp(random.uniform(math.log(ratio[0]), math.log(ratio[1])))
        new_w = int(round(math.sqrt(target_area * aspect_ratio)))
        new_h = int(round(math.sqrt(target_area / aspect_ratio)))
        if new_w <= image.width and new_h <= image.height:
            top = random.randint(0, image.height - new_h)
            left = random.randint(0, image.width - new_w)
            image = F.resized_crop(image, top, left, new_h, new_w, size)
            mask = F.resized_crop(mask, top, left, new_h, new_w, size, interpolation=F.InterpolationMode.NEAREST)
            return image, mask
    return F.resize(image, size), F.resize(mask, size, interpolation=F.InterpolationMode.NEAREST)


def seg_augment(image, mask, cfg: SegAugCfg):
    if not cfg.enable:
        image = F.resize(image, (cfg.image_size, cfg.image_size))
        mask = F.resize(mask, (cfg.image_size, cfg.image_size), interpolation=F.InterpolationMode.NEAREST)
        return image, mask
    image, mask = seg_random_crop_pair(image, mask, (cfg.image_size, cfg.image_size))
    if random.random() < 0.5:
        image = F.hflip(image)
        mask = F.hflip(mask)
    if random.random() < 0.3:
        pts = [(random.randint(-12,12), random.randint(-12,12)) for _ in range(4)]
        image = F.perspective(image, [(0,0),(image.width-1,0),(image.width-1,image.height-1),(0,image.height-1)], pts, interpolation=F.InterpolationMode.BILINEAR)
        mask = F.perspective(mask, [(0,0),(mask.width-1,0),(mask.width-1,mask.height-1),(0,mask.height-1)], pts, interpolation=F.InterpolationMode.NEAREST)
    image = F.adjust_brightness(image, 1.0 + random.uniform(-0.2, 0.2))
    image = F.adjust_contrast(image, 1.0 + random.uniform(-0.2, 0.2))
    image = F.adjust_saturation(image, 1.0 + random.uniform(-0.15, 0.15))
    return image, mask


def seg_to_tensor(image, mask):
    image = F.to_tensor(image)
    image = F.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    mask = F.to_tensor(mask)
    return image, (mask > 0.5).float()


def seg_filter(hf_dataset, max_samples):
    ds = hf_dataset.filter(lambda x: x['label'] == 2 and x['mask'] is not None, load_from_cache_file=False)
    if max_samples is not None:
        ds = ds.select(range(min(max_samples, len(ds))))
    return ds


class SegDataset(Dataset):
    def __init__(self, hf_dataset, image_size, augment=False):
        self.dataset = hf_dataset
        self.cfg = SegAugCfg(image_size=image_size, enable=augment)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        record = self.dataset[idx]
        image = record['image'].convert('RGB')
        mask = record['mask']
        if mask.mode != 'L':
            mask = mask.convert('L')
        image, mask = seg_augment(image, mask, self.cfg)
        image, mask = seg_to_tensor(image, mask)
        return {'image': image, 'mask': mask}



### B.4 DataLoaders & Preview

In [ ]:
hf_train = seg_filter(load_dataset(SEG_CFG['dataset_name'], split='train', streaming=False), SEG_CFG['train_samples'])
hf_val_full = seg_filter(load_dataset(SEG_CFG['dataset_name'], split='validation', streaming=False), SEG_CFG['val_samples'] + SEG_CFG['test_samples'])
hf_val = hf_val_full.select(range(SEG_CFG['val_samples']))
hf_test = hf_val_full.select(range(SEG_CFG['val_samples'], SEG_CFG['val_samples'] + SEG_CFG['test_samples']))

train_ds_seg = SegDataset(hf_train, SEG_CFG['image_size'], augment=SEG_CFG['augment'])
val_ds_seg = SegDataset(hf_val, SEG_CFG['image_size'], augment=False)
test_ds_seg = SegDataset(hf_test, SEG_CFG['image_size'], augment=False)

train_loader_seg = DataLoader(train_ds_seg, batch_size=SEG_CFG['batch_size'], shuffle=True, num_workers=SEG_CFG['num_workers'])
val_loader_seg = DataLoader(val_ds_seg, batch_size=SEG_CFG['batch_size'], shuffle=False, num_workers=SEG_CFG['num_workers'])
test_loader_seg = DataLoader(test_ds_seg, batch_size=SEG_CFG['batch_size'], shuffle=False, num_workers=SEG_CFG['num_workers'])

print('Segmentation dataloaders -> train:', len(train_loader_seg), 'val:', len(val_loader_seg), 'test:', len(test_loader_seg))

batch = next(iter(train_loader_seg))
fig, axes = plt.subplots(2, 4, figsize=(10,5))
for i in range(4):
    img = batch['image'][i]
    mask = batch['mask'][i]
    img_disp = img * torch.tensor([0.229, 0.224, 0.225]).view(3,1,1) + torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    axes[0, i].imshow(np.transpose(img_disp.clamp(0,1).numpy(), (1,2,0)))
    axes[0, i].axis('off')
    axes[0, i].set_title('Image')
    axes[1, i].imshow(mask.squeeze(0).numpy(), cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title('Mask')
plt.tight_layout(); plt.show()



### B.5 Model & Optimiser

In [ ]:
class UNet(nn.Module):
    def __init__(self, base_width=32):
        super().__init__()
        b = base_width
        self.enc1 = SegConvBlock(3, b)
        self.enc2 = SegConvBlock(b, b*2)
        self.enc3 = SegConvBlock(b*2, b*4)
        self.enc4 = SegConvBlock(b*4, b*8)
        self.pool = nn.MaxPool2d(2)
        self.center = SegConvBlock(b*8, b*16)
        self.up4 = nn.ConvTranspose2d(b*16, b*8, 2, stride=2)
        self.dec4 = SegConvBlock(b*16, b*8)
        self.up3 = nn.ConvTranspose2d(b*8, b*4, 2, stride=2)
        self.dec3 = SegConvBlock(b*8, b*4)
        self.up2 = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.dec2 = SegConvBlock(b*4, b*2)
        self.up1 = nn.ConvTranspose2d(b*2, b, 2, stride=2)
        self.dec1 = SegConvBlock(b*2, b)
        self.head = nn.Conv2d(b, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        c = self.center(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(c), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)


def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    num = (probs * targets).sum(dim=(1,2,3))
    den = probs.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3))
    return 1 - (2*num + eps)/(den + eps)

def dice_coefficient(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()
    num = (preds * targets).sum(dim=(1,2,3))
    den = preds.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3))
    return (2*num + eps)/(den + eps)

def iou_coefficient(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()
    intersection = (preds * targets).sum(dim=(1,2,3))
    union = preds.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3)) - intersection
    return (intersection + eps)/(union + eps)

model_seg = UNet(base_width=32).to(SEG_DEVICE)
criterion_bce_seg = nn.BCEWithLogitsLoss()
optimizer_seg = optim.AdamW(model_seg.parameters(), lr=SEG_CFG['learning_rate'], weight_decay=1e-4)
scaler_seg = amp.amp.GradScaler('cuda', 'cuda', enabled=(SEG_DEVICE.type == 'cuda'))
scheduler_seg = optim.lr_scheduler.CosineAnnealingLR(optimizer_seg, T_max=SEG_CFG['epochs']) if SEG_CFG['use_cosine_schedule'] else None
ema_seg = UNet(base_width=32).to(SEG_DEVICE)
ema_seg.load_state_dict(model_seg.state_dict())
for param in ema_seg.parameters():
    param.requires_grad_(False)



### B.6 Training Loop

In [ ]:
history_seg = defaultdict(list)
best_state_seg = None
best_val_dice = -math.inf

for epoch in range(SEG_CFG['epochs']):
    model_seg.train()
    train_loss = 0.0
    train_dice = 0.0
    batches = 0

    for batch in train_loader_seg:
        images = batch['image'].to(SEG_DEVICE)
        masks = batch['mask'].to(SEG_DEVICE)

        optimizer_seg.zero_grad(set_to_none=True)
        with amp.autocast(device_type='cuda', enabled=(SEG_DEVICE.type == 'cuda')):
            logits = model_seg(images)
            bce = criterion_bce_seg(logits, masks)
            dice = dice_loss(logits, masks).mean()
            loss = 0.5*bce + 0.5*dice

        scaler_seg.scale(loss).backward()
        if SEG_CFG['grad_clip_norm'] > 0:
            scaler_seg.unscale_(optimizer_seg)
            torch.nn.utils.clip_grad_norm_(model_seg.parameters(), SEG_CFG['grad_clip_norm'])
        scaler_seg.step(optimizer_seg)
        scaler_seg.update()

        if SEG_CFG['ema_decay'] > 0:
            for ema_param, param in zip(ema_seg.parameters(), model_seg.parameters()):
                ema_param.data.mul_(SEG_CFG['ema_decay']).add_(param.data, alpha=1 - SEG_CFG['ema_decay'])

        train_loss += loss.item()
        train_dice += dice_coefficient(logits, masks).mean().item()
        batches += 1

    if scheduler_seg is not None:
        scheduler_seg.step()

    model_seg.eval()
    ema_seg.eval()
    val_loss = 0.0
    val_dice = 0.0
    val_iou = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader_seg:
            images = batch['image'].to(SEG_DEVICE)
            masks = batch['mask'].to(SEG_DEVICE)
            logits = ema_seg(images)
            bce = criterion_bce_seg(logits, masks)
            dice = dice_loss(logits, masks).mean()
            loss = 0.5*bce + 0.5*dice
            val_loss += loss.item()
            val_dice += dice_coefficient(logits, masks).mean().item()
            val_iou += iou_coefficient(logits, masks).mean().item()
            val_batches += 1

    train_loss /= max(batches, 1)
    train_dice /= max(batches, 1)
    val_loss /= max(val_batches, 1)
    val_dice /= max(val_batches, 1)
    val_iou /= max(val_batches, 1)

    history_seg['train_loss'].append(train_loss)
    history_seg['train_dice'].append(train_dice)
    history_seg['val_loss'].append(val_loss)
    history_seg['val_dice'].append(val_dice)
    history_seg['val_iou'].append(val_iou)
    history_seg['lr'].append(optimizer_seg.param_groups[0]['lr'])

    print(f"[Segmentation] Epoch {epoch+1:02d}/{SEG_CFG['epochs']} - train_loss={train_loss:.3f} train_dice={train_dice:.3f} val_loss={val_loss:.3f} val_dice={val_dice:.3f} val_iou={val_iou:.3f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_state_seg = {k: v.clone() for k, v in ema_seg.state_dict().items()}

if best_state_seg is not None:
    ema_seg.load_state_dict(best_state_seg)



### B.7 Evaluation & Visualisations

In [ ]:
model_path_seg = SEG_RUN_DIR / 'best_model.pth'
history_path_seg = SEG_RUN_DIR / 'history.json'
torch.save(ema_seg.state_dict(), model_path_seg)
with history_path_seg.open('w') as fp:
    json.dump({k: v for k, v in history_seg.items()}, fp, indent=2)
print('Saved segmentation model to', model_path_seg)

dice_scores = []
iou_scores = []
examples = []
ema_seg.eval()
with torch.no_grad():
    for batch in test_loader_seg:
        images = batch['image'].to(SEG_DEVICE)
        masks = batch['mask'].to(SEG_DEVICE)
        logits = ema_seg(images)
        dice_scores.extend(dice_coefficient(logits, masks).cpu().tolist())
        iou_scores.extend(iou_coefficient(logits, masks).cpu().tolist())
        preds = (torch.sigmoid(logits) > 0.5).float()
        for img, mask, pred in zip(images[:3], masks[:3], preds[:3]):
            examples.append((img.cpu(), mask.cpu(), pred.cpu()))

(SEG_RUN_DIR / 'evaluation.json').write_text(json.dumps({'dice_scores': dice_scores, 'iou_scores': iou_scores}, indent=2))
print(f"Segmentation dice {np.mean(dice_scores):.3f} ± {np.std(dice_scores):.3f}")
print(f"Segmentation IoU  {np.mean(iou_scores):.3f} ± {np.std(iou_scores):.3f}")

plt.figure(figsize=(8,4))
plt.plot(history_seg['train_loss'], label='Train')
plt.plot(history_seg['val_loss'], label='Validation')
plt.title('Segmentation Loss Curves')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()

plt.figure(figsize=(8,4))
plt.plot(history_seg['train_dice'], label='Train Dice')
plt.plot(history_seg['val_dice'], label='Val Dice')
plt.title('Segmentation Dice Curves')
plt.xlabel('Epoch'); plt.ylabel('Dice'); plt.legend(); plt.show()

plt.figure(figsize=(8,4))
plt.plot(history_seg['val_iou'], label='Val IoU')
plt.title('Segmentation IoU Curve')
plt.xlabel('Epoch'); plt.ylabel('IoU'); plt.legend(); plt.show()

plt.figure(figsize=(8,4))
plt.plot(history_seg['lr'])
plt.title('Segmentation Learning Rate Schedule')
plt.xlabel('Epoch'); plt.ylabel('LR'); plt.show()

rows = len(examples)
fig, axes = plt.subplots(rows, 3, figsize=(9, 3*rows))
if rows == 1:
    axes = np.expand_dims(axes, axis=0)
for row, (img, mask, pred) in enumerate(examples):
    img_disp = img * torch.tensor([0.229, 0.224, 0.225]).view(3,1,1) + torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    axes[row, 0].imshow(np.transpose(img_disp.clamp(0,1).numpy(), (1,2,0)))
    axes[row, 0].set_title('Image')
    axes[row, 1].imshow(mask.squeeze(0).numpy(), cmap='gray')
    axes[row, 1].set_title('Mask')
    axes[row, 2].imshow(pred.squeeze(0).numpy(), cmap='gray')
    axes[row, 2].set_title('Prediction')
    for ax in axes[row]:
        ax.axis('off')
plt.tight_layout(); plt.show()

print('Segmentation artifacts:')
print('
'.join(sorted(p.name for p in SEG_RUN_DIR.iterdir())))

